# Signature Recognition & Forgery Detection
## Data Preprocessing

This notebook prepares the SVC 2004 Task 2 online signature data
for machine learning and deep learning models.

### Preprocessing Pipeline
1. Load signature data
2. Create labels and metadata
3. Convert timestamp to elapsed time
4. Normalize features
5. Split data by user
6. Pad variable-length sequences
7. Prepare training, validation and testing datasets

## 1. Import libraries

In [2]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Dataset path

In [3]:
dataset_path = r"C:\Users\saksh\Downloads\Task2\Task2"

print("Dataset exists:", os.path.exists(dataset_path))

Dataset exists: True


## 3. Get all TXT files

In [4]:
txt_files = [
    file for file in os.listdir(dataset_path)
    if file.lower().endswith(".txt")
]

print("Total signature files:", len(txt_files))

Total signature files: 1600


## 4. Creating a reusable loader

In [5]:
def load_signature(file_path):
    """
    Load one SVC 2004 signature TXT file.

    First line  = number of points
    Remaining lines = 7 signature features
    """

    with open(file_path, "r") as f:
        lines = f.readlines()

    num_points = int(lines[0].strip())

    data = [
        line.strip().split()
        for line in lines[1:]
        if line.strip()
    ]

    columns = ["x","y","timestamp","pen_status","azimuth","altitude","pressure"
    ]

    df = pd.DataFrame(data, columns=columns)
    df = df.astype(float)

    # Verify number of points
    if len(df) != num_points:
        print(
            f"Warning: expected {num_points} points, "
            f"but found {len(df)}"
        )

    return df

## 5. Test our loader

In [6]:
sample_path = os.path.join(dataset_path, "U10S1.TXT")
sample_signature = load_signature(sample_path)

print("Shape:", sample_signature.shape)

display(sample_signature.head())

Shape: (142, 7)


,x,y,timestamp,pen_status,azimuth,altitude,pressure
0,3236.0,5028.0,17638839.0,0.0,1260.0,400.0,351.0
1,3428.0,4998.0,17638849.0,1.0,1260.0,400.0,382.0
2,3541.0,5056.0,17638859.0,1.0,1260.0,400.0,394.0
3,3665.0,5135.0,17638869.0,1.0,1310.0,400.0,397.0
4,3945.0,5229.0,17638879.0,1.0,1370.0,390.0,405.0


## 6. Create metadata & labels

In [7]:
metadata = []
for file in txt_files:

    match = re.match(r"U(\d+)S(\d+)", file.upper())

    if match:
        user_id = int(match.group(1))
        signature_id = int(match.group(2))

        # S1-S20 → Genuine
        # S21-S40 → Forged
        label = 0 if signature_id <= 20 else 1

        metadata.append({
            "file": file,"user_id": user_id,"signature_id": signature_id,"label": label
        })

metadata_df = pd.DataFrame(metadata)

print("Metadata shape:", metadata_df.shape)

display(metadata_df.head())

Metadata shape: (1600, 4)


,file,user_id,signature_id,label
0,U10S1.TXT,10,1,0
1,U10S10.TXT,10,10,0
2,U10S11.TXT,10,11,0
3,U10S12.TXT,10,12,0
4,U10S13.TXT,10,13,0


## 7. Convert timestamp-->>0

In [8]:
def preprocess_signature(df):
    df = df.copy()

    # Convert absolute timestamp into elapsed time
    df["timestamp"] = (
        df["timestamp"] - df["timestamp"].iloc[0]
    )

    return df

In [9]:
processed_sample = preprocess_signature(sample_signature)

display(processed_sample.head())

,x,y,timestamp,pen_status,azimuth,altitude,pressure
0,3236.0,5028.0,0.0,0.0,1260.0,400.0,351.0
1,3428.0,4998.0,10.0,1.0,1260.0,400.0,382.0
2,3541.0,5056.0,20.0,1.0,1260.0,400.0,394.0
3,3665.0,5135.0,30.0,1.0,1310.0,400.0,397.0
4,3945.0,5229.0,40.0,1.0,1370.0,390.0,405.0


## 8. Check timestamp

In [10]:
print(
    "Original first timestamp:",
    sample_signature["timestamp"].iloc[0]
)

print(
    "Processed first timestamp:",
    processed_sample["timestamp"].iloc[0]
)

print(
    "Processed last timestamp:",
    processed_sample["timestamp"].iloc[-1]
)

Original first timestamp: 17638839.0
Processed first timestamp: 0.0
Processed last timestamp: 1541.0


In [11]:
sample = preprocess_signature(sample_signature)

print("Type:", type(sample))
print("Shape:", sample.shape)
print(sample[:5])

Type: <class 'pandas.core.frame.DataFrame'>
Shape: (142, 7)
        x       y  timestamp  pen_status  azimuth  altitude  pressure
0  3236.0  5028.0        0.0         0.0   1260.0     400.0     351.0
1  3428.0  4998.0       10.0         1.0   1260.0     400.0     382.0
2  3541.0  5056.0       20.0         1.0   1260.0     400.0     394.0
3  3665.0  5135.0       30.0         1.0   1310.0     400.0     397.0
4  3945.0  5229.0       40.0         1.0   1370.0     390.0     405.0


In [12]:
#Fix the sequence length first
sequence_lengths = []

for _, row in metadata_df.iterrows():
    file_path = os.path.join(dataset_path, row["file"])
    
    signature = load_signature(file_path)
    processed = preprocess_signature(signature)
    
    sequence_lengths.append(processed.shape[0])

print("Minimum length:", min(sequence_lengths))
print("Maximum length:", max(sequence_lengths))
print("Unique lengths:", len(set(sequence_lengths)))

Minimum length: 80
Maximum length: 713
Unique lengths: 337


In [13]:
from collections import Counter

length_counts = Counter(sequence_lengths)

print("Most common lengths:")
print(length_counts.most_common(10))

Most common lengths:
[(168, 18), (133, 17), (181, 17), (169, 17), (176, 17), (183, 17), (221, 16), (175, 16), (170, 16), (186, 15)]


In [14]:
print("Minimum:", min(sequence_lengths))
print("Maximum:", max(sequence_lengths))

print("Median:", np.median(sequence_lengths))
print("Mean:", np.mean(sequence_lengths))

Minimum: 80
Maximum: 713
Median: 187.0
Mean: 208.208125


In [15]:
print("Sequences <= 200:", sum(np.array(sequence_lengths) <= 200))
print("Sequences > 200:", sum(np.array(sequence_lengths) > 200))
print("Percentage > 200:", round(
    sum(np.array(sequence_lengths) > 200) / len(sequence_lengths) * 100, 2
), "%")

Sequences <= 200: 933
Sequences > 200: 667
Percentage > 200: 41.69 %


In [16]:
print("Sequences <= 250:", sum(np.array(sequence_lengths) <= 250))
print("Sequences > 250:", sum(np.array(sequence_lengths) > 250))
print("Percentage > 250:", round(
    sum(np.array(sequence_lengths) > 250) / len(sequence_lengths) * 100, 2
), "%")

Sequences <= 250: 1281
Sequences > 250: 319
Percentage > 250: 19.94 %


In [17]:
print("Sequences <= 300:", sum(np.array(sequence_lengths) <= 300))
print("Sequences > 300:", sum(np.array(sequence_lengths) > 300))
print("Percentage > 300:", round(
    sum(np.array(sequence_lengths) > 300) / len(sequence_lengths) * 100, 2
), "%")

Sequences <= 300: 1425
Sequences > 300: 175
Percentage > 300: 10.94 %


## 10. Padding + Truncation

In [18]:
X = []
y = []

for _, row in metadata_df.iterrows():
    file_path = os.path.join(dataset_path, row["file"])
    
    signature = load_signature(file_path)
    processed = preprocess_signature(signature)
    
    X.append(processed)
    y.append(row["label"])

X = np.array([
    np.pad(
        s[:300],
        ((0, max(0, 300 - len(s))), (0, 0)),
        mode="constant"
    )
    for s in X
], dtype=np.float32)

y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1600, 300, 7)
y shape: (1600,)


## 9. Train/Test Split & normalization

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1280, 300, 7)
X_test: (320, 300, 7)
y_train: (1280,)
y_test: (320,)


In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

X_train_scaled: (1280, 300, 7)
X_test_scaled: (320, 300, 7)


## 10. Start RNN

In [21]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PyTorch version: 2.13.0+cpu
Device: cpu


In [22]:
#NumPy → PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print("X_train:", X_train_tensor.shape)
print("X_test:", X_test_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("y_test:", y_test_tensor.shape)

X_train: torch.Size([1280, 300, 7])
X_test: torch.Size([320, 300, 7])
y_train: torch.Size([1280])
y_test: torch.Size([320])


In [23]:
#DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 40
Testing batches: 10


In [24]:
#RNN Model
class SignatureRNN(nn.Module):
    def __init__(self, input_size=7, hidden_size=64, num_layers=1):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,hidden_size=hidden_size,num_layers=num_layers,batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, hidden = self.rnn(x)

        # Last timestep
        out = out[:, -1, :]

        out = self.fc(out)

        return out.squeeze(1)

In [25]:
#Create model 
model = SignatureRNN().to(device)

print(model)

SignatureRNN(
  (rnn): RNN(7, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [26]:
#Loss & optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [27]:
#Training 
epochs = 20

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {avg_loss:.4f}"
    )

Epoch [1/20], Loss: 0.6868
Epoch [2/20], Loss: 0.6850
Epoch [3/20], Loss: 0.6840
Epoch [4/20], Loss: 0.6835
Epoch [5/20], Loss: 0.6845
Epoch [6/20], Loss: 0.6830
Epoch [7/20], Loss: 0.6832
Epoch [8/20], Loss: 0.6820
Epoch [9/20], Loss: 0.6811
Epoch [10/20], Loss: 0.6821
Epoch [11/20], Loss: 0.6816
Epoch [12/20], Loss: 0.6822
Epoch [13/20], Loss: 0.6812
Epoch [14/20], Loss: 0.6815
Epoch [15/20], Loss: 0.6800
Epoch [16/20], Loss: 0.6813
Epoch [17/20], Loss: 0.6804
Epoch [18/20], Loss: 0.6786
Epoch [19/20], Loss: 0.6781
Epoch [20/20], Loss: 0.6779


In [28]:
#Evaluate accuracy
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        outputs = model(batch_X)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        correct += (predictions == batch_y).sum().item()
        total += batch_y.size(0)

accuracy = correct / total

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 56.56%


In [29]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)

        outputs = model(batch_X)
        predictions = (torch.sigmoid(outputs) >= 0.5).cpu().numpy()

        all_preds.extend(predictions.astype(int))
        all_labels.extend(batch_y.numpy().astype(int))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=["Genuine", "Forged"]
))

Confusion Matrix:
[[153   7]
 [132  28]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.54      0.96      0.69       160
      Forged       0.80      0.17      0.29       160

    accuracy                           0.57       320
   macro avg       0.67      0.57      0.49       320
weighted avg       0.67      0.57      0.49       320



In [30]:
print("Genuine:", np.sum(y_test == 0))
print("Forged:", np.sum(y_test == 1))

Genuine: 160
Forged: 160


## 11. Data Sanity Check

In [33]:
print("Train labels:")
print(np.bincount(y_train.astype(int)))

print("\nTest labels:")
print(np.bincount(y_test.astype(int)))

Train labels:
[640 640]

Test labels:
[160 160]


In [34]:
print("Original sequences:", len(sequence_lengths))
print("Sequences > 300:", sum(np.array(sequence_lengths) > 300))

Original sequences: 1600
Sequences > 300: 175


In [35]:
print("Train labels:")
print(np.bincount(y_train.astype(int)))

print("\nTest labels:")
print(np.bincount(y_test.astype(int)))

Train labels:
[640 640]

Test labels:
[160 160]


In [36]:
train_indices, test_indices = train_test_split(
    np.arange(len(metadata_df)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

train_users = set(metadata_df.iloc[train_indices]["user_id"])
test_users = set(metadata_df.iloc[test_indices]["user_id"])

print("Train users:", len(train_users))
print("Test users:", len(test_users))
print("Common users:", len(train_users & test_users))

Train users: 40
Test users: 40
Common users: 40


In [37]:
print(
    metadata_df.groupby(["user_id", "label"])
    .size()
    .unstack(fill_value=0)
)

label     0   1
user_id        
1        20  20
2        20  20
3        20  20
4        20  20
5        20  20
6        20  20
7        20  20
8        20  20
9        20  20
10       20  20
11       20  20
12       20  20
13       20  20
14       20  20
15       20  20
16       20  20
17       20  20
18       20  20
19       20  20
20       20  20
21       20  20
22       20  20
23       20  20
24       20  20
25       20  20
26       20  20
27       20  20
28       20  20
29       20  20
30       20  20
31       20  20
32       20  20
33       20  20
34       20  20
35       20  20
36       20  20
37       20  20
38       20  20
39       20  20
40       20  20


In [38]:
print(metadata_df.groupby("label")["user_id"].nunique())

label
0    40
1    40
Name: user_id, dtype: int64


In [43]:
print(processed_sample.describe())

print(processed_sample.isna().sum())

                 x            y    timestamp  pen_status      azimuth  \
count   142.000000   142.000000   142.000000  142.000000   142.000000   
mean   6500.690141  4168.239437   783.274648    0.978873  1373.521127   
std    1409.807135  1024.779173   466.168191    0.144316    59.527658   
min    3236.000000  1596.000000     0.000000    0.000000  1260.000000   
25%    5368.500000  3699.250000   352.500000    1.000000  1340.000000   
50%    6692.000000  4130.000000   836.000000    1.000000  1375.000000   
75%    7809.250000  4712.500000  1188.500000    1.000000  1420.000000   
max    8484.000000  6375.000000  1541.000000    1.000000  1470.000000   

         altitude    pressure  
count  142.000000  142.000000  
mean   552.746479  544.788732  
std     58.831944  146.882103  
min    370.000000   10.000000  
25%    550.000000  418.500000  
50%    570.000000  591.500000  
75%    580.000000  655.750000  
max    630.000000  761.000000  
x             0
y             0
timestamp     0
pen_st

In [45]:
print("Current X_train_scaled:", X_train_scaled.shape)
print("Current X_test_scaled:", X_test_scaled.shape)

Current X_train_scaled: (1280, 300, 7)
Current X_test_scaled: (320, 300, 7)


In [46]:
from sklearn.preprocessing import StandardScaler

X_clean = []
y_clean = []

for _, row in metadata_df.iterrows():
    file_path = os.path.join(dataset_path, row["file"])
    
    signature = load_signature(file_path)
    processed = preprocess_signature(signature).astype(np.float32)

    # Normalize each feature within the actual sequence
    scaler_local = StandardScaler()
    processed = scaler_local.fit_transform(processed)

    # Pad/truncate after normalization
    if len(processed) < 300:
        padded = np.zeros((300, 7), dtype=np.float32)
        padded[:len(processed)] = processed
    else:
        padded = processed[:300]

    X_clean.append(padded)
    y_clean.append(row["label"])

X_clean = np.array(X_clean, dtype=np.float32)
y_clean = np.array(y_clean)

print("X_clean:", X_clean.shape)
print("y_clean:", y_clean.shape)

X_clean: (1600, 300, 7)
y_clean: (1600,)


In [47]:
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean,
    y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean
)

print("X_train_clean:", X_train_clean.shape)
print("X_test_clean:", X_test_clean.shape)
print("y_train_clean:", y_train_clean.shape)
print("y_test_clean:", y_test_clean.shape)

X_train_clean: (1280, 300, 7)
X_test_clean: (320, 300, 7)
y_train_clean: (1280,)
y_test_clean: (320,)


In [48]:
X_train_tensor = torch.tensor(X_train_clean, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_clean, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_clean, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_clean, dtype=torch.float32)

print(X_train_tensor.shape)
print(X_test_tensor.shape)
print(y_train_tensor.shape)
print(y_test_tensor.shape)

torch.Size([1280, 300, 7])
torch.Size([320, 300, 7])
torch.Size([1280])
torch.Size([320])


In [49]:
train_dataset_clean = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset_clean = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader_clean = DataLoader(
    train_dataset_clean,
    batch_size=32,
    shuffle=True
)

test_loader_clean = DataLoader(
    test_dataset_clean,
    batch_size=32,
    shuffle=False
)

print("Training batches:", len(train_loader_clean))
print("Testing batches:", len(test_loader_clean))

Training batches: 40
Testing batches: 10


## 12. LSTM

In [50]:
lstm_clean = SignatureLSTM().to(device)

criterion_clean = nn.BCEWithLogitsLoss()

optimizer_clean = torch.optim.Adam(
    lstm_clean.parameters(),
    lr=0.001
)

print(lstm_clean)

SignatureLSTM(
  (lstm): LSTM(7, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


## 13. Train

In [51]:
epochs = 20

for epoch in range(epochs):

    lstm_clean.train()
    total_loss = 0

    for batch_X, batch_y in train_loader_clean:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer_clean.zero_grad()

        outputs = lstm_clean(batch_X)

        loss = criterion_clean(outputs, batch_y)

        loss.backward()
        optimizer_clean.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader_clean)

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/20], Loss: 0.6904
Epoch [2/20], Loss: 0.6824
Epoch [3/20], Loss: 0.6765
Epoch [4/20], Loss: 0.6727
Epoch [5/20], Loss: 0.6681
Epoch [6/20], Loss: 0.6654
Epoch [7/20], Loss: 0.6653
Epoch [8/20], Loss: 0.6683
Epoch [9/20], Loss: 0.6633
Epoch [10/20], Loss: 0.6597
Epoch [11/20], Loss: 0.6569
Epoch [12/20], Loss: 0.6543
Epoch [13/20], Loss: 0.6514
Epoch [14/20], Loss: 0.6494
Epoch [15/20], Loss: 0.6511
Epoch [16/20], Loss: 0.6445
Epoch [17/20], Loss: 0.6424
Epoch [18/20], Loss: 0.6415
Epoch [19/20], Loss: 0.6380
Epoch [20/20], Loss: 0.6375


In [52]:
lstm_clean.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader_clean:

        batch_X = batch_X.to(device)

        outputs = lstm_clean(batch_X)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).int().cpu().numpy()

        all_preds.extend(predictions)
        all_labels.extend(batch_y.numpy().astype(int))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=["Genuine", "Forged"]
))

Confusion Matrix:
[[158   2]
 [140  20]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.53      0.99      0.69       160
      Forged       0.91      0.12      0.22       160

    accuracy                           0.56       320
   macro avg       0.72      0.56      0.45       320
weighted avg       0.72      0.56      0.45       320



In [55]:
print(metadata_df.groupby("label").size())

print("\nUsers:")
print(metadata_df["user_id"].nunique())

print("\nSignatures per user:")
print(metadata_df.groupby("user_id").size().describe())

print(metadata_df.groupby(["user_id", "label"]).size().head(20))

label
0    800
1    800
dtype: int64

Users:
40

Signatures per user:
count    40.0
mean     40.0
std       0.0
min      40.0
25%      40.0
50%      40.0
75%      40.0
max      40.0
dtype: float64
user_id  label
1        0        20
         1        20
2        0        20
         1        20
3        0        20
         1        20
4        0        20
         1        20
5        0        20
         1        20
6        0        20
         1        20
7        0        20
         1        20
8        0        20
         1        20
9        0        20
         1        20
10       0        20
         1        20
dtype: int64


In [56]:
print(
    metadata_df.groupby(["user_id", "label"]).size()
    .unstack(fill_value=0)
)

label     0   1
user_id        
1        20  20
2        20  20
3        20  20
4        20  20
5        20  20
6        20  20
7        20  20
8        20  20
9        20  20
10       20  20
11       20  20
12       20  20
13       20  20
14       20  20
15       20  20
16       20  20
17       20  20
18       20  20
19       20  20
20       20  20
21       20  20
22       20  20
23       20  20
24       20  20
25       20  20
26       20  20
27       20  20
28       20  20
29       20  20
30       20  20
31       20  20
32       20  20
33       20  20
34       20  20
35       20  20
36       20  20
37       20  20
38       20  20
39       20  20
40       20  20


In [57]:
print("Genuine feature means:")
print(X[y == 0].mean(axis=(0, 1)))

print("\nForged feature means:")
print(X[y == 1].mean(axis=(0, 1)))

print("\nDifference:")
print(
    X[y == 0].mean(axis=(0, 1))
    -
    X[y == 1].mean(axis=(0, 1))
)

Genuine feature means:
[3.5176379e+03 2.7888613e+03 7.7275598e+02 5.7910001e-01 7.6162183e+02
 3.3254608e+02 3.8113342e+02]

Forged feature means:
[3.9803342e+03 3.4929895e+03 1.1197351e+03 7.0147914e-01 9.3236493e+02
 3.8786270e+02 3.6956360e+02]

Difference:
[-4.62696289e+02 -7.04128174e+02 -3.46979126e+02 -1.22379124e-01
 -1.70743103e+02 -5.53166199e+01  1.15698242e+01]


In [58]:
valid_means = []

for i, length in enumerate(sequence_lengths):
    valid_length = min(length, 300)
    valid_means.append(X[i, :valid_length, :].mean(axis=0))

valid_means = np.array(valid_means)

print("Genuine feature means:")
print(valid_means[y == 0].mean(axis=0))

print("\nForged feature means:")
print(valid_means[y == 1].mean(axis=0))

print("\nDifference:")
print(
    valid_means[y == 0].mean(axis=0)
    - valid_means[y == 1].mean(axis=0)
)

Genuine feature means:
[5.9360156e+03 4.6721226e+03 1.1782518e+03 9.7565329e-01 1.2886971e+03
 5.6001794e+02 6.3614685e+02]

Forged feature means:
[5.6140371e+03 4.8751436e+03 1.4681327e+03 9.8150098e-01 1.3103121e+03
 5.4387128e+02 5.1601440e+02]

Difference:
[ 3.2197852e+02 -2.0302100e+02 -2.8988086e+02 -5.8476925e-03
 -2.1614990e+01  1.6146667e+01  1.2013245e+02]


## 13A. Feature Engineering

In [59]:
features = []

for i, length in enumerate(sequence_lengths):
    length = min(length, 300)
    seq = X[i, :length, :]

    mean_features = seq.mean(axis=0)
    std_features = seq.std(axis=0)

    features.append(
        np.concatenate([mean_features, std_features])
    )

features = np.array(features, dtype=np.float32)

print("Feature matrix:", features.shape)

Feature matrix: (1600, 14)


In [60]:
X_feat_train, X_feat_test, y_feat_train, y_feat_test = train_test_split(
    features,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_feat_train.shape)
print(X_feat_test.shape)

(1280, 14)
(320, 14)


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

clf = LogisticRegression(max_iter=2000)

clf.fit(X_feat_train, y_feat_train)

pred = clf.predict(X_feat_test)

print("Confusion Matrix:")
print(confusion_matrix(y_feat_test, pred))

print("\nClassification Report:")
print(classification_report(
    y_feat_test,
    pred,
    target_names=["Genuine", "Forged"]
))

## 13B. Properly scaled Logistic Regression

In [62]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=5000,
        random_state=42
    ))
])

lr_model.fit(X_feat_train, y_feat_train)

pred = lr_model.predict(X_feat_test)

print("Confusion Matrix:")
print(confusion_matrix(y_feat_test, pred))

print("\nClassification Report:")
print(classification_report(
    y_feat_test,
    pred,
    target_names=["Genuine", "Forged"]
))

Confusion Matrix:
[[123  37]
 [ 47 113]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.72      0.77      0.75       160
      Forged       0.75      0.71      0.73       160

    accuracy                           0.74       320
   macro avg       0.74      0.74      0.74       320
weighted avg       0.74      0.74      0.74       320



## 14. Random Forest

In [63]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_feat_train, y_feat_train)

rf_pred = rf_model.predict(X_feat_test)

print("Confusion Matrix:")
print(confusion_matrix(y_feat_test, rf_pred))

print("\nClassification Report:")
print(classification_report(
    y_feat_test,
    rf_pred,
    target_names=["Genuine", "Forged"]
))

Confusion Matrix:
[[151   9]
 [ 12 148]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.93      0.94      0.93       160
      Forged       0.94      0.93      0.93       160

    accuracy                           0.93       320
   macro avg       0.93      0.93      0.93       320
weighted avg       0.93      0.93      0.93       320



## 15A. User-independent evaluation

In [64]:
from sklearn.model_selection import GroupShuffleSplit

groups = metadata_df["user_id"].values

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(features, y, groups=groups)
)

X_group_train = features[train_idx]
X_group_test = features[test_idx]

y_group_train = y[train_idx]
y_group_test = y[test_idx]

print("Train samples:", len(train_idx))
print("Test samples:", len(test_idx))
print("Train users:", metadata_df.iloc[train_idx]["user_id"].nunique())
print("Test users:", metadata_df.iloc[test_idx]["user_id"].nunique())
print(
    "Common users:",
    len(
        set(metadata_df.iloc[train_idx]["user_id"])
        &
        set(metadata_df.iloc[test_idx]["user_id"])
    )
)

Train samples: 1280
Test samples: 320
Train users: 32
Test users: 8
Common users: 0


## 15B. Random forest on unseen user

rf_group = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_group.fit(X_group_train, y_group_train)

rf_group_pred = rf_group.predict(X_group_test)

print("Confusion Matrix:")
print(confusion_matrix(y_group_test, rf_group_pred))

print("\nClassification Report:")
print(classification_report(
    y_group_test,
    rf_group_pred,
    target_names=["Genuine", "Forged"]
))

## 16. Random Forest tuning

In [66]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_group_train, y_group_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV F1:")
print(grid_search.best_score_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Parameters:
{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}

Best CV F1:
0.7036183217891712


In [67]:
best_rf = grid_search.best_estimator_

best_pred = best_rf.predict(X_group_test)

print("Confusion Matrix:")
print(confusion_matrix(y_group_test, best_pred))

print("\nClassification Report:")
print(classification_report(
    y_group_test,
    best_pred,
    target_names=["Genuine", "Forged"]
))

Confusion Matrix:
[[128  32]
 [ 18 142]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.88      0.80      0.84       160
      Forged       0.82      0.89      0.85       160

    accuracy                           0.84       320
   macro avg       0.85      0.84      0.84       320
weighted avg       0.85      0.84      0.84       320



## 17. Feature Importance 🌳

In [68]:
feature_names = [
    "mean_x", "mean_y", "mean_timestamp",
    "mean_pen_status", "mean_azimuth",
    "mean_altitude", "mean_pressure",
    "std_x", "std_y", "std_timestamp",
    "std_pen_status", "std_azimuth",
    "std_altitude", "std_pressure"
]

importance = best_rf.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance_df)

            Feature  Importance
6     mean_pressure    0.127579
9     std_timestamp    0.118359
2    mean_timestamp    0.100830
4      mean_azimuth    0.077484
12     std_altitude    0.073634
10   std_pen_status    0.072174
3   mean_pen_status    0.066949
1            mean_y    0.057689
7             std_x    0.055138
5     mean_altitude    0.054790
13     std_pressure    0.053496
8             std_y    0.049015
11      std_azimuth    0.047223
0            mean_x    0.045640


## 18. Final Model Save 💾

In [69]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(
    rf_group,
    "models/signature_random_forest.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [70]:
joblib.dump(
    feature_names,
    "models/feature_names.pkl"
)

print("Feature names saved!")

Feature names saved!


In [71]:
print(os.path.exists("models/signature_random_forest.pkl"))
print(os.path.exists("models/feature_names.pkl"))

True
True


## 19A. Prediction function

In [72]:
def predict_signature(file_path):

    # Load signature
    signature = load_signature(file_path)

    # Preprocess
    processed = preprocess_signature(signature)

    # Use maximum 300 timesteps
    processed = processed.iloc[:300]

    # Calculate 14 features
    mean_features = processed.mean(axis=0).values
    std_features = processed.std(axis=0).values

    features = np.concatenate([
        mean_features,
        std_features
    ]).reshape(1, -1)

    # Load saved model
    model = joblib.load(
        "models/signature_random_forest.pkl"
    )

    # Prediction
    prediction = model.predict(features)[0]

    # Probability
    probabilities = model.predict_proba(features)[0]

    label = "Genuine" if prediction == 0 else "Forged"

    confidence = probabilities[prediction] * 100

    print("Prediction:", label)
    print(f"Confidence: {confidence:.2f}%")

    return label, confidence

## 19B. Known Genuine signature test

In [74]:
genuine_file = metadata_df[
    metadata_df["label"] == 0
].iloc[0]["file"]

genuine_path = os.path.join(
    dataset_path,
    genuine_file
)

print("Testing file:", genuine_file)

predict_signature(genuine_path)


forged_file = metadata_df[
    metadata_df["label"] == 1
].iloc[0]["file"]

forged_path = os.path.join(
    dataset_path,
    forged_file
)

print("Testing file:", forged_file)

predict_signature(forged_path)

Testing file: U10S1.TXT
Prediction: Genuine
Confidence: 90.33%
Testing file: U10S21.TXT
Prediction: Forged
Confidence: 94.33%


('Forged', np.float64(94.33333333333334))

## 20. Test on ALL unseen users

In [75]:
final_pred = rf_group.predict(X_group_test)

final_accuracy = (final_pred == y_group_test).mean()

print("========== FINAL MODEL PERFORMANCE ==========")
print(f"Accuracy: {final_accuracy * 100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_group_test, final_pred))

print("\nClassification Report:")
print(classification_report(
    y_group_test,
    final_pred,
    target_names=["Genuine", "Forged"]
))

========== FINAL MODEL PERFORMANCE ==========
Accuracy: 84.69%

Confusion Matrix:
[[124  36]
 [ 13 147]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.91      0.78      0.84       160
      Forged       0.80      0.92      0.86       160

    accuracy                           0.85       320
   macro avg       0.85      0.85      0.85       320
weighted avg       0.85      0.85      0.85       320



In [76]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

from preprocessing import preprocess_signature
from feature_engineering import extract_features

print("Source modules imported successfully!")

Source modules imported successfully!
